In [152]:
import pandas as pd
import numpy as np
import re
import difflib
from cleaning import extract_refresh_rate
from cleaning import extract_camera_info
from cleaning import add_ram
from cleaning import clean_phone_name
from cleaning import clean_metrics
from cleaning import clean_price
from cleaning import clean_storage
from cleaning import clean_chipset
from cleaning import add_chipset_info


In [153]:
cps = pd.read_csv(r'cellphones_full.csv')
cps.info()

<class 'pandas.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   Tên                    966 non-null    str  
 1   Giá                    966 non-null    str  
 2   Link                   966 non-null    str  
 3   Kích thước màn hình    865 non-null    str  
 4   Công nghệ màn hình     806 non-null    str  
 5   Camera sau             847 non-null    str  
 6   Camera trước           817 non-null    str  
 7   Chipset                850 non-null    str  
 8   Công nghệ NFC          763 non-null    str  
 9   Bộ nhớ trong           913 non-null    str  
 10  Thẻ SIM                695 non-null    str  
 11  Hệ điều hành           756 non-null    str  
 12  Độ phân giải màn hình  657 non-null    str  
 13  Tính năng màn hình     725 non-null    str  
 14  Loại CPU               586 non-null    str  
 15  Dung lượng RAM         870 non-null    str  
 16  P

ĐỐI TÊN THUỘC TÍNH

In [154]:
feature_mapping = {
    "Tên": "Name",
    "Giá": "Price",
    "Link": "Link",
    "Kích thước màn hình": "Screen Size",
    "Công nghệ màn hình": "Display",
    "Camera sau": "Rear Camera",
    "Camera trước": "Front Camera",
    "Chipset": "Chipset",
    "Công nghệ NFC": "NFC",
    "Bộ nhớ trong": "ROM",
    "Thẻ SIM": "SIM Card",
    "Hệ điều hành": "Operating System",
    "Độ phân giải màn hình": "Screen Resolution",
    "Tính năng màn hình": "Display Features",
    "Loại CPU": "CPU",
    "Dung lượng RAM": "RAM",
    "Pin": "Battery",
    "Tương thích": "Compatibility",
    "Cảm biến": "Sensors",
}

cps = cps.rename(columns=feature_mapping)


In [155]:
df = cps.copy()

Đọc từ gsm và antutu

In [156]:
gsm = pd.read_csv(r'all_phones_final.csv')
gsm = gsm[['name_clean', 'Memory | Internal']]

In [157]:
antutu = pd.read_csv(r'antutu_score_socket.csv')
att = antutu.copy()

EXTRACT REFRESH RATE

In [158]:
df["Refresh Rate"] = df["Display Features"].apply(extract_refresh_rate)

In [159]:
df["Refresh Rate"].info()

<class 'pandas.Series'>
RangeIndex: 966 entries, 0 to 965
Series name: Refresh Rate
Non-Null Count  Dtype  
--------------  -----  
569 non-null    float64
dtypes: float64(1)
memory usage: 7.7 KB


CLEAN NAME

In [160]:
df["Name"] = df["Name"].apply(clean_phone_name)
gsm['name_clean'] = gsm['name_clean'].apply(clean_phone_name)

ADD RAM

In [161]:
df = add_ram(gsm, df)

In [162]:
df['RAM'].info()

<class 'pandas.Series'>
RangeIndex: 966 entries, 0 to 965
Series name: RAM
Non-Null Count  Dtype
--------------  -----
891 non-null    str  
dtypes: str(1)
memory usage: 7.7 KB


CLEAN CHIPSET AND ADD CHIPSET INFO

In [163]:
df['Chipset'] = df['Chipset'].apply(clean_chipset)
df = add_chipset_info(att, df)

In [164]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Name               966 non-null    str    
 1   Price              966 non-null    str    
 2   Link               966 non-null    str    
 3   Screen Size        865 non-null    str    
 4   Display            806 non-null    str    
 5   Rear Camera        847 non-null    str    
 6   Front Camera       817 non-null    str    
 7   Chipset            834 non-null    str    
 8   NFC                763 non-null    str    
 9   ROM                913 non-null    str    
 10  SIM Card           695 non-null    str    
 11  Operating System   756 non-null    str    
 12  Screen Resolution  657 non-null    str    
 13  Display Features   725 non-null    str    
 14  CPU                586 non-null    str    
 15  RAM                891 non-null    str    
 16  Battery            861 non-null    st

CLEAN PRICE

In [165]:
df["Price"] = df["Price"].apply(clean_price)

In [166]:
df["Price"].info()

<class 'pandas.Series'>
RangeIndex: 966 entries, 0 to 965
Series name: Price
Non-Null Count  Dtype  
--------------  -----  
337 non-null    float64
dtypes: float64(1)
memory usage: 7.7 KB


CLEAN STORAGE

In [167]:
df["RAM"] = df["RAM"].apply(clean_storage)
df["ROM"] = df["ROM"].apply(clean_storage)

In [168]:
df[["RAM", "ROM"]].info()

<class 'pandas.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   RAM     891 non-null    float64
 1   ROM     913 non-null    float64
dtypes: float64(2)
memory usage: 15.2 KB


CLEAN METRICS

In [169]:
cols_to_clean = ["Screen Size","Battery", "clock"]
for col in cols_to_clean:
    df[col] = df[col].apply(clean_metrics)

In [170]:
df[["Screen Size","Battery", "clock"]].info()

<class 'pandas.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Screen Size  865 non-null    float64
 1   Battery      859 non-null    float64
 2   clock        747 non-null    float64
dtypes: float64(3)
memory usage: 22.8 KB


CLEAN CPU


In [171]:
WORD_CORE_MAP = {
    'dual': 2, 'quad': 4, 'hexa': 6, 'octa': 8,
    'deca': 10, 'nona': 9,
    'tám nhân': 8, 'lõi tám': 8, 'tám lõi': 8,
    'lõi tứ': 4, 'lõi đơn': 1,
}

def normalize(t):
    t = str(t).lower()
    t = re.sub(r'(\d)[,،，](\d)', r'\1.\2', t)
    t = t.replace('，', ',')
    return t

def extract_groups(t):
    patterns = [
        r'(\d+)\s*[x×*]\s*(\d+\.?\d*)\s*ghz',
        r'(\d+)\s*[x×*]\s*[\w\s.\-]+?@\s*(\d+\.?\d*)\s*ghz',
        r'(\d+)\s*[x×*]\s*[\w\s.\-]+?(?:up to|tối đa|lên đến|đến)\s*(\d+\.?\d*)\s*ghz',
        r'(\d+)\s*[x×]\s*[\w\s.\-]+?\((?:tối đa\s*)?(\d+\.?\d*)\s*ghz\)',
        r'(\d+)\s*(?:nhân|lõi)\s+(\d+\.?\d*)\s*ghz',
        r'(\d+)\s*[x×]\s*[a-z]\w+\s+(\d+\.?\d*)\s*ghz',
    ]
    for pat in patterns:
        found = re.findall(pat, t)
        if len(found) >= 2:
            valid = [(int(m[0]), float(m[-1])) for m in found if float(m[-1]) > 0.5]
            if len(valid) >= 2:
                return valid
    return []

def extract_single_freq(t):
    freqs = re.findall(r'(\d+\.?\d*)\s*ghz', t)
    freqs = [float(f) for f in freqs if float(f) > 0.5]
    return max(freqs) if freqs else np.nan

def extract_cpu_features(raw):
    if pd.isna(raw) or str(raw).strip() == '':
        return {}
    t = normalize(raw)
    result = {}

    for word, num in WORD_CORE_MAP.items():
        if word in t:
            result['num_cores'] = num
            break
    if 'num_cores' not in result:
        m = re.search(r'(\d+)\s*(nhân|cores?|lõi)', t)
        result['num_cores'] = int(m.group(1)) if m else np.nan

    groups = extract_groups(t)
    if groups:
        groups_sorted = sorted(groups, key=lambda x: x[1], reverse=True)
        result['perf_cores']    = groups_sorted[0][0]
        result['perf_freq_ghz'] = groups_sorted[0][1]
        result['eff_cores']     = groups_sorted[-1][0]
        result['eff_freq_ghz']  = groups_sorted[-1][1]
        # Cộng tổng từ groups nếu num_cores chưa có
        if pd.isna(result.get('num_cores')):
            result['num_cores'] = sum(g[0] for g in groups)
    else:
        single = extract_single_freq(t)
        if not np.isnan(single):
            result['max_freq_ghz'] = single

    if 'perf_cores' not in result:
        m = re.search(r'(\d+)\s*lõi\s*(?:hiệu năng|hiệu suất)', t)
        if m: result['perf_cores'] = int(m.group(1))
    if 'eff_cores' not in result:
        m = re.search(r'(\d+)\s*lõi\s*(?:tiết kiệm|nhỏ)', t)
        if m: result['eff_cores'] = int(m.group(1))

    return result


In [172]:
features = df['CPU'].apply(extract_cpu_features)
cpu_df   = pd.json_normalize(features)
df       = pd.concat([df.reset_index(drop=True), cpu_df], axis=1)

CLEAN OPERATING SYSTEM

In [173]:
def advanced_clean_os(text):
    if (
        pd.isna(text)
        or not isinstance(text, str)
        or "cập nhật" in text.lower()
    ):
        return 1, "Android", None

    text = text.strip()
    
    is_android = 1
    os_name = "Android"
    if "ios" in text.lower():
        is_android = 0
        os_name = "iOS"

    # trích xuất số đời (Version)
    os_version = None

    # Trường hợp A: Dòng chỉ chứa mỗi số (Ví dụ: '11')
    if text.isdigit():
        return is_android, os_name, float(text)

    # Trường hợp B: Dòng phức tạp có chữ 'có thể nâng cấp lên Android X'
    if "nâng cấp" in text.lower():
        upgraded_version = re.findall(r"Android\s*(\d+(?:\.\d+)?)", text)
        if upgraded_version:
            # Lấy số phiên bản cuối cùng (cao nhất) trong chuỗi
            return is_android, os_name, float(upgraded_version[-1])

    # Trường hợp C: Dòng thông thường, tìm số đi ngay sau chữ 'Android' hoặc 'iOS'
    version_match = re.search(r"(?:Android|iOS)\s*(\d+(?:\.\d+)?)", text, re.I)
    if version_match:
        os_version = float(version_match.group(1))
    else:
        # Trường hợp như không có số, tạm để None hoặc gán số 8.0/9.0
        os_version = None

    return is_android, os_name, os_version

In [174]:
df["OS_Is_Android"], df["OS_Name"], \
    df["OS_Version"] = zip(*df["Operating System"].apply(advanced_clean_os))

CLEAN RESOLUTION

In [175]:
def extract_res_row(text):
    # Nếu dòng bị trống (NaN) hoặc không phải chữ
    if pd.isna(text) or not isinstance(text, str):
        return None, None

    # Tìm cấu trúc a x b ở đầu dòng
    match = re.search(r"^(\d+)\s*[xX×]\s*(\d+)", text.strip())

    if match:
        # Trả về một Tuple gồm (Width, Height) kiểu số nguyên
        return int(match.group(1)), int(match.group(2))

    return None, None

In [176]:
df["Reso_Width"], df["Reso_Height"] = zip(
    *df["Screen Resolution"].apply(extract_res_row)
)

In [177]:
def clean_sim_options(text):
    max_nano = 0
    max_esim = 0
    max_micro = 0
    max_mini = 0

    if pd.isna(text) or not isinstance(text, str):
        return max_nano, max_esim, max_micro, max_mini

    text_lower = text.lower()

    options = re.split(r"hoặc|/|;", text_lower)

    for option in options:
        option = option.strip()

        nano_in_opt = 0
        esim_in_opt = 0

        # Xử lý eSIM 
        if "esim" in option:
            match_esim = re.search(r"(\d+)\s*esim", option)
            if match_esim:
                esim_in_opt = int(match_esim.group(1))
            elif "dual" in option or "kép" in option:
                esim_in_opt = 2
            else:
                esim_in_opt = 1

        # Xử lý Nano SIM 
        if "nano" in option or "sim 1 + sim 2" in option:
            match_nano = re.search(r"(\d+)\s*nano", option)
            if match_nano:
                nano_in_opt = int(match_nano.group(1))
            elif (
                "dual" in option
                or "kép" in option
                or "sim 1 + sim 2" in option
            ):
                nano_in_opt = 2
            else:
                nano_in_opt = 1
        elif "2 sim" in option and "nano" in text_lower:
            nano_in_opt = 2
        # Trường hợp ghi mỗi chữ "Nano-SIM" thuần túy
        elif "nano" in option:
            nano_in_opt = 1

        # Cập nhật giá trị max
        max_nano = max(max_nano, nano_in_opt)
        max_esim = max(max_esim, esim_in_opt)

        # Xử lý các loại SIM cổ (Mini, Micro)
        if "micro" in option:
            max_micro = 1
        if "mini" in option:
            max_mini = 1

    return max_nano, max_esim, max_micro, max_mini

In [178]:
(
    df["Nano_SIM_Count"],
    df["eSIM_Count"],
    df["Micro_SIM_Count"],
    df["Mini_SIM_Count"],
) = zip(*df["SIM Card"].apply(clean_sim_options))

In [179]:
df[['CPU', 'num_cores', 'perf_cores', 'eff_cores', 'perf_freq_ghz', 'eff_freq_ghz', 'max_freq_ghz']]

,CPU,num_cores,perf_cores,eff_cores,perf_freq_ghz,eff_freq_ghz,max_freq_ghz
0,CPU 6 lõi với 2 lõi hiệu năng và 4 lõi tiết ki...,6.0,2.0,4.0,NaN,NaN,NaN
1,8 nhân,8.0,NaN,NaN,NaN,NaN,NaN
2,CPU 6 lõi với 2 lõi hiệu năng và 4 lõi tiết ki...,6.0,2.0,4.0,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
961,2 nhân 2.5 GHz & 6 nhân 2.0 GHz,2.0,2.0,6.0,2.5,2.0,NaN
962,1×Cortex-A710 2.5GHz + 3×Cortex-A710 2.36GHz +...,NaN,NaN,NaN,NaN,NaN,2.5
963,NaN,NaN,NaN,NaN,NaN,NaN,NaN
964,NaN,NaN,NaN,NaN,NaN,NaN,NaN


CLEAN NFC

In [180]:
df['NFC'] = df['NFC'].map(lambda x : 1 if x == "Có" else 0)

CLEAN CAMERA

In [181]:
df = extract_camera_info(df)

In [182]:
df.head()

,Name,Price,Link,Screen Size,Display,Rear Camera,Front Camera,Chipset,NFC,ROM,...,Micro_SIM_Count,Mini_SIM_Count,rear_count,rear_mp_max,rear_f/,rear_ois,rear_telephoto,rear_wide,front_mp,front_f/
0,iphone 17,34590000.0,https://cellphones.com.vn/iphone-17-pro.html,6.30,Super Retina XDR,Chính: 48MP khẩu độ ƒ/1.6 OIS hỗ trợ chụp 24MP...,Camera 18MP Center Stage Khẩu độ ƒ/1.9,a19 pro,1,256.0,...,0,0,3.0,48.0,1.6,1.0,1.0,1.0,18.0,1.9
1,oppo find x9s,23990000.0,https://cellphones.com.vn/dien-thoai-oppo-find...,6.59,AMOLED,Góc siêu rộng: 50MP; Góc rộng: 50MP; Telephoto...,32MP,dimensity 9500s,1,256.0,...,0,0,3.0,50.0,0.0,0.0,1.0,1.0,32.0,NaN
2,iphone 17 promax,36990000.0,https://cellphones.com.vn/iphone-17-pro-max.html,6.90,Super Retina XDR,Chính: 48MP khẩu độ ƒ/1.6 OIS hỗ trợ chụp 24MP...,Camera 18MP Center Stage Khẩu độ ƒ/1.9,a19 pro,1,256.0,...,0,0,3.0,48.0,1.6,1.0,1.0,1.0,18.0,1.9
3,samsung galaxy s26,30490000.0,https://cellphones.com.vn/dien-thoai-samsung-g...,6.90,Dynamic AMOLED 2X,Camera siêu rộng: 50MPCamera góc rộng: 200MPCa...,12MP,snapdragon 8 elite gen 5,1,256.0,...,0,0,4.0,200.0,0.0,0.0,1.0,1.0,12.0,NaN
4,samsung galaxy s26,20490000.0,https://cellphones.com.vn/dien-thoai-samsung-g...,6.30,Dynamic AMOLED 2X,Camera siêu rộng: 12MPCamera góc rộng: 50MPCam...,12MP,exynos 2600,1,256.0,...,0,0,3.0,50.0,0.0,0.0,1.0,1.0,12.0,NaN


In [183]:
# camera = pd.read_csv('camera_score.csv')
# camera = camera.rename(columns={'camera_score': 'Camera_score'})

In [184]:
# camera['name'] = camera['name'].str.lower()

# df = pd.merge(camera, df, left_on='name', right_on='Name', how='right')

# df = df.drop(columns=['name'])

In [185]:
# # Bước 1: Tách cột đó ra khỏi DataFrame trước (để tránh bị trùng lặp)
# score_col = df.pop('Camera_score')

# # Bước 2: Chèn lại vào vị trí mong muốn (Ví dụ: loc=2 là cột thứ 3 trong bảng)
# df.insert(loc=46, column='Camera_score', value=score_col)

Filter những điện thoại còn đầy đủ thông số nhất

In [186]:
# df = df[
#     df['antutu_11'].notna() & (
#         df['Chipset'].notna() |
#         df['Reso_Width'].notna() |
#         df['Display'].notna()
#     )
# ].copy()

# print(len(df))  # → 668

In [187]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Data columns (total 46 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Name               966 non-null    str    
 1   Price              337 non-null    float64
 2   Link               966 non-null    str    
 3   Screen Size        865 non-null    float64
 4   Display            806 non-null    str    
 5   Rear Camera        847 non-null    str    
 6   Front Camera       817 non-null    str    
 7   Chipset            834 non-null    str    
 8   NFC                966 non-null    int64  
 9   ROM                913 non-null    float64
 10  SIM Card           695 non-null    str    
 11  Operating System   756 non-null    str    
 12  Screen Resolution  657 non-null    str    
 13  Display Features   725 non-null    str    
 14  CPU                586 non-null    str    
 15  RAM                891 non-null    float64
 16  Battery            859 non-null    fl

In [188]:
features_with_null = df.columns[df.isnull().any()].tolist()
features_with_null

['Price',
 'Screen Size',
 'Display',
 'Rear Camera',
 'Front Camera',
 'Chipset',
 'ROM',
 'SIM Card',
 'Operating System',
 'Screen Resolution',
 'Display Features',
 'CPU',
 'RAM',
 'Battery',
 'Compatibility',
 'Sensors',
 'Refresh Rate',
 'antutu_11',
 'clock',
 'gpu',
 'num_cores',
 'perf_cores',
 'eff_cores',
 'perf_freq_ghz',
 'eff_freq_ghz',
 'max_freq_ghz',
 'OS_Version',
 'Reso_Width',
 'Reso_Height',
 'rear_count',
 'rear_mp_max',
 'rear_f/',
 'rear_ois',
 'rear_telephoto',
 'rear_wide',
 'front_mp',
 'front_f/']

Encode OS name. 1: Android and 0: iOS

In [189]:
# Nếu là Android thì thành 1, ngược lại (iOS) thì thành 0
df['OS_Name'] = (df['OS_Name'] == 'Android').astype(int)

Encode Display. 1: OLED/AMOLED, 2: IPS LCD/IPS/LCD, 0: other

In [190]:
display_lower = df['Display'].astype(str).str.lower()

# 2. Định nghĩa các điều kiện (Conditions)
conditions = [
    display_lower.str.contains('oled|amoled', regex=True),  # Nhóm 2
    display_lower.str.contains('ips lcd|ips|lcd', regex=True)      # Nhóm 1
]

# 3. Định nghĩa giá trị tương ứng cho từng điều kiện
choices = [2, 1]

# 4. Sử dụng np.select, mặc định (default) không khớp cái nào sẽ là 0
df['Display'] = np.select(conditions, choices, default=0)

Encode Chipset. Apple: 0, Snapdragon: 1, MediaTek: 2, Exynos: 3, Kirin: 4, Unisoc: 5, Other: 6

In [191]:
def encode_chipset(val):
    val = str(val).lower()
    if 'apple' in val or 'chip a' in val or 'bionic' in val:
        return 'Apple'
    elif 'snapdragon' in val or 'qualcomm' in val:
        return 'Snapdragon'
    elif 'dimensity' in val or 'helio' in val or 'mediatek' in val:
        return 'MediaTek'
    elif 'exynos' in val:
        return 'Exynos'
    elif 'kirin' in val:
        return 'Kirin'
    elif 'unisoc' in val:
        return 'Unisoc'
    else:
        return 'Other'
 
brand_map = {'Apple': 0, 'Snapdragon': 1, 'MediaTek': 2,
             'Exynos': 3, 'Kirin': 4, 'Unisoc': 5, 'Other': 6}
df['Chipset'] = df['Chipset'].apply(encode_chipset).map(brand_map)

Encode GPU

In [192]:
def encode_gpu(val):
    val = str(val).lower()
    if 'adreno'     in val: return 'Adreno'
    elif 'mali'     in val: return 'Mali'
    elif 'apple'    in val: return 'Apple GPU'
    elif 'xclipse'  in val: return 'Xclipse'
    elif 'powervr'  in val or 'img' in val: return 'PowerVR'
    elif 'immortalis' in val: return 'Immortalis'
    elif 'maleoon'  in val: return 'Maleoon'
    else: return 'Other'
 
gpu_map = {'Adreno': 0, 'Mali': 1, 'Apple GPU': 2, 'Xclipse': 3,
           'PowerVR': 4, 'Immortalis': 5, 'Maleoon': 6, 'Other': 7}
df['gpu_family_enc'] = df['gpu'].apply(encode_gpu).map(gpu_map)

Encode Reso

In [193]:
median_screen = df.loc[df['Screen Size'] > 0, 'Screen Size'].median()
df['Screen Size'] = df['Screen Size'].replace(0, median_screen)
 
df['PPI'] = (
    np.sqrt(df['Reso_Width']**2 + df['Reso_Height']**2) / df['Screen Size']
).round(1)

Derive PPI

In [194]:
median_screen = df.loc[df['Screen Size'] > 0, 'Screen Size'].median()
df['Screen Size'] = df['Screen Size'].replace(0, median_screen)

df['PPI'] = (
    np.sqrt(df['Reso_Width']**2 + df['Reso_Height']**2) / df['Screen Size']
).round(1)

df.drop(columns=['Reso_Width', 'Reso_Height'], inplace=True)


Derive SIM_total

In [195]:
df['SIM_total'] = (
    df['Nano_SIM_Count'] +
    df['eSIM_Count'] +
    df['Micro_SIM_Count'] +
    df['Mini_SIM_Count']
)

df.drop(columns=['Nano_SIM_Count', 'Micro_SIM_Count', 'Mini_SIM_Count'], inplace=True)



Derive eSIM, dien thoai nao co eSIM: 1, khong co: 0

In [196]:
df['has_eSIM'] = (df['eSIM_Count'] > 0).astype(int)

df.drop(columns=['eSIM_Count'], inplace=True)


In [197]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Data columns (total 44 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Name               966 non-null    str    
 1   Price              337 non-null    float64
 2   Link               966 non-null    str    
 3   Screen Size        865 non-null    float64
 4   Display            966 non-null    int64  
 5   Rear Camera        847 non-null    str    
 6   Front Camera       817 non-null    str    
 7   Chipset            966 non-null    int64  
 8   NFC                966 non-null    int64  
 9   ROM                913 non-null    float64
 10  SIM Card           695 non-null    str    
 11  Operating System   756 non-null    str    
 12  Screen Resolution  657 non-null    str    
 13  Display Features   725 non-null    str    
 14  CPU                586 non-null    str    
 15  RAM                891 non-null    float64
 16  Battery            859 non-null    fl

In [198]:
# # Helper function
# def fill_median_group(df, col, group_col):
#     df[col] = df.groupby(group_col)[col].transform(
#         lambda x: x.fillna(x.median())
#     )
#     df[col] = df[col].fillna(df[col].median())  # fallback
#     return df

# # ── Nhóm CPU: groupby theo chipset_brand_enc ──────────────────
# for col in ['perf_cores', 'eff_cores', 'perf_freq_ghz', 'eff_freq_ghz']:
#     df = fill_median_group(df, col, 'Chipset')

# # ── OS_Version: groupby theo os_is_ios ───────────────────────
# df = fill_median_group(df, 'OS_Version', 'OS_Name')

# # ── PPI: groupby theo display_tier ───────────────────────────
# df = fill_median_group(df, 'PPI', 'Display')

# # Kiểm tra
# print(df[['perf_cores','eff_cores','perf_freq_ghz','eff_freq_ghz'
#           ,'OS_Version','Camera_score','PPI']].isnull().sum())

In [199]:
drop = [
    'Link', 'Rear Camera', 'Front Camera', 'CPU', 'Display Features', 'Screen Resolution',
    'Operating System', 'SIM Card', 'OS_Is_Android', 'num_cores', 'Compatibility', 'Sensors', 'max_freq_ghz']

In [200]:
df.drop(columns=drop, inplace = True)

In [201]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Data columns (total 31 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Name            966 non-null    str    
 1   Price           337 non-null    float64
 2   Screen Size     865 non-null    float64
 3   Display         966 non-null    int64  
 4   Chipset         966 non-null    int64  
 5   NFC             966 non-null    int64  
 6   ROM             913 non-null    float64
 7   RAM             891 non-null    float64
 8   Battery         859 non-null    float64
 9   Refresh Rate    569 non-null    float64
 10  antutu_11       747 non-null    float64
 11  clock           747 non-null    float64
 12  gpu             747 non-null    str    
 13  perf_cores      395 non-null    float64
 14  eff_cores       389 non-null    float64
 15  perf_freq_ghz   365 non-null    float64
 16  eff_freq_ghz    365 non-null    float64
 17  OS_Name         966 non-null    int64  
 18  O

In [202]:
df.to_csv('full_data_2.csv')